# 04 — Industry Data Preparation

This notebook prepares additional industry-level indicators used in the forecasting dataset.

These inputs complement the ACEA registration and OECD macroeconomic data, vehicle market structure indicators where available.

The final outputs are standardized by country and month so they can be merged into the master modelling dataset.

## Data Source Note

This notebook was originally built using a proprietary dataset from GlobalData ("World Tyre Forecast Service").

Due to licensing restrictions, the dataset is not included in this repository.

The code is provided for demonstration purposes and can be adapted to similar industry datasets with comparable structure.

In [ ]:
import pandas as pd

file_path = r"C:\Users\Source_file.xlsx"

df_raw = pd.read_excel(
    file_path,
    sheet_name=0,
    header=[6, 7, 8]
)

display(df_raw.head())
print(df_raw.columns)

In [ ]:
INDUSTRY_RAW_DIR = BRONZE_DIR / "industry"

file_path = INDUSTRY_RAW_DIR / "Source_file.xlsx"

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver" / "industry"

INDUSTRY_RAW_DIR = BRONZE_DIR / "industry"

SILVER_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

df = df_raw.copy()

# Build new column names from:
# - second header level: year
# - third header level: quarter
# - first data row: measure / id label
new_cols = []

for i, col in enumerate(df.columns):
    year_part = str(col[1]).strip() if len(col) > 1 and pd.notna(col[1]) else ""
    quarter_part = str(col[2]).strip() if len(col) > 2 and pd.notna(col[2]) else ""
    row0_part = str(df.iloc[0, i]).strip() if pd.notna(df.iloc[0, i]) else ""

    # id columns
    if row0_part in ["Grand Total", "Region", "Country"]:
        new_cols.append(row0_part)
    else:
        new_cols.append(f"{year_part}_{quarter_part}_{row0_part}")

df.columns = new_cols

# drop the first row because it was used as header content
df = df.iloc[1:].copy()

# rename ids
df = df.rename(columns={
    "Grand Total": "grand_total",
    "Region": "region",
    "Country": "country"
})

print(df.columns.tolist()[:20])

## Reshape Quarterly Industry Indicators

This section flattens the multi-level Excel header, reshapes the data from wide to long format, filters the relevant indicators, and pivots the output into one row per country-quarter.

In [ ]:
id_cols = ["grand_total", "region", "country"]
value_cols = [c for c in df.columns if c not in id_cols]

long_df = df.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="period_measure",
    value_name="value"
)

# split only columns that really match year_quarter_measure
split_df = long_df["period_measure"].str.extract(
    r"^(?P<year>\d{4})_(?P<quarter>Q[1-4])_(?P<measure>.+)$"
)

long_df = pd.concat([long_df, split_df], axis=1)

print(long_df[["period_measure", "year", "quarter", "measure"]].head(20))
print(long_df["measure"].dropna().unique()[:20])

In [ ]:
target_measures = ["Vehicle Production", "Vehicle Parc"]

# 1. Filter on needed measures

filtered_df = long_df[
    long_df["measure"].isin(target_measures)
].copy()

# 2. Filter on needed period (2022 - 2025)

filtered_df["year"] = pd.to_numeric(filtered_df["year"], errors="coerce")
filtered_df["value"] = pd.to_numeric(filtered_df["value"], errors="coerce")

filtered_df = filtered_df[
    filtered_df["year"].notna() & filtered_df["value"].notna()
].copy()

filtered_df["year"] = filtered_df["year"].astype(int)
filtered_df = filtered_df[filtered_df["year"].between(2021, 2025)].copy()

filtered_df["time_period"] = filtered_df["year"].astype(str) + "-" + filtered_df["quarter"]
filtered_df["date"] = pd.PeriodIndex(filtered_df["time_period"], freq="Q").to_timestamp()

# --------------------------------------------------
# Additional filtering / relabeling
# --------------------------------------------------

# 3. Remove rows where country = 'Totals'
filtered_df = filtered_df[filtered_df["country"] != "Totals"].copy()

# 4. Rename region 'Totals' -> 'Total Europe'
filtered_df.loc[filtered_df["region"] == "Totals", "region"] = "Total Europe"

# 5. Where region was Totals and country is missing, set country = 'European Union'
filtered_df.loc[
    (filtered_df["region"] == "Total Europe") & (filtered_df["country"].isna()),
    "country"
] = "European Union"

final_df = filtered_df[
    ["grand_total", "region", "country", "year", "quarter", "time_period", "date", "measure", "value"]
].sort_values(["country", "year", "quarter", "measure"])

display(final_df.head(5))
print(final_df.shape)
print(final_df["measure"].value_counts())

In [ ]:
# --------------------------------------------------
# Pivot measures into separate columns
# --------------------------------------------------

final_df = (
    filtered_df
    .pivot_table(
        index=[ "region", "country", "year", "quarter", "time_period", "date"],
        columns="measure",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Remove the pivoted columns index name
final_df.columns.name = None

# Optional: rename columns to cleaner names
final_df = final_df.rename(columns={
    "Vehicle Production": "vehicle_production",
    "Vehicle Parc": "vehicle_parc"
})

# Sort for readability
final_df = final_df.sort_values(["country", "year", "quarter"]).reset_index(drop=True)

display(final_df.head(5))


In [ ]:
# --------------------------------------------------
# Calculate YoY growth for vehicle parc
# --------------------------------------------------

final_df = final_df.sort_values(["country", "date"])

final_df["vehicle_parc_yoy"] = (
    final_df
    .groupby("country")["vehicle_parc"]
    .pct_change(4) * 100
)

In [ ]:
#  Remove rows where year != '2022 - 2025' 

final_df = final_df[
    final_df["year"].between(2022, 2025)
].copy()

In [ ]:
# Expanding the data to monthly granularity  

gd_auto_df = final_df.copy()

expanded_rows = []

for _, row in gd_auto_df.iterrows():
    quarter_start = pd.Period(row["time_period"], freq="Q").start_time

    for month_offset in range(3):
        new_row = row.copy()
        new_row["date"] = quarter_start + pd.DateOffset(months=month_offset)
        new_row["year"] = new_row["date"].year
        new_row["month"] = new_row["date"].month
        new_row["quarter"] = f"Q{new_row['date'].quarter}"
        expanded_rows.append(new_row)

gd_auto_monthly_df = pd.DataFrame(expanded_rows)

# Optional: sort nicely
gd_auto_monthly_df = gd_auto_monthly_df.sort_values(
    ["country", "date"]
).reset_index(drop=True)



In [ ]:
country_to_code_mapping = {
    "Austria": "AT",
    "Belgium/Lux": "BE",
    "Bulgaria": "BG",
    "Cyprus/Malta": "CY",
    "Czech": "CZ",
    "Germany": "DE",
    "Denmark": "DK",
    "Estonia": "EE",
    "European Union": "EU",
    "Euro Zone": "EUR",
    "Spain": "ES",
    "Finland": "FI",
    "France": "FR",
    "Greece": "GR",
    "Croatia": "HR",
    "Hungary": "HU",
    "Ireland": "IE",
    "Italy": "IT",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Latvia": "LV",
    "Malta": "MT",
    "Netherlands": "NL",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Sweden": "SE",
    "Slovenia": "SI",
    "Slovakia": "SK"
}

In [ ]:
gd_auto_monthly_df["country_code"] = gd_auto_monthly_df["country"].map(country_to_code_mapping)

In [ ]:
# =========================================================
#  Saving all dataframes to desk 
# =========================================================


import os

# Create folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save datasets
gd_auto_monthly_df.to_parquet("data/gd_auto_monthly_df.parquet", index=False)
